# Depth Extraction from a Monocular Video using Depth Anything 3 (DA3METRIC‑LARGE)

This notebook demonstrates how to extract per‑frame depth maps from a monocular video using the **DA3METRIC‑LARGE** model from the Depth Anything 3 project. It is designed for local execution on an **M2 MacBook Pro** (or similar) without requiring a cloud GPU. The pipeline:

1. **Install** the necessary libraries and clone the Depth Anything 3 repository.
2. **Define the path** to your input video (MP4), extract frames at a reasonable frame rate for depth estimation.
3. **Load the DA3METRIC‑LARGE model** and run inference on the frames using Apple’s MPS (Metal Performance Shaders) backend when available, or CPU as a fallback.
4. **Save** raw depth maps (`.npy`) and normalized visualization images (`.png`) for each frame.

> **Note**: Running the model on Apple Silicon (M1/M2/M3) is slower than using a dedicated NVIDIA GPU. Adjust the target FPS, resolution, or batch size to suit your hardware and patience.


In [ ]:
# Install dependencies and clone Depth Anything 3
# Run this cell once. If packages are already installed, you can skip re‑running it.

# Clone the repository (will do nothing if it already exists)
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3.git || true
%cd Depth-Anything-3

# Install required Python packages.
# Note: xformers may not provide wheels for Apple Silicon; installation will silently continue if unavailable.
!pip install --quiet torch torchvision
!pip install --quiet xformers || true
!pip install --quiet opencv-python pillow matplotlib tqdm imageio imageio-ffmpeg
!pip install --quiet -e .


In [ ]:
import os
import cv2
from tqdm import tqdm

# Define the path to your input video.
# Change this to the full path of your MP4 file on your Mac.
video_path = "/path/to/your/video.mp4"  # <-- EDIT THIS

# Directory to save extracted frames
FRAME_DIR = "frames"
os.makedirs(FRAME_DIR, exist_ok=True)

# Target frame rate for extraction (frames per second).
# Lower values mean fewer frames and faster processing.
TARGET_FPS = 3

# Extract frames from the video
cap = cv2.VideoCapture(video_path)
src_fps = cap.get(cv2.CAP_PROP_FPS)
if src_fps <= 0:
    src_fps = 30  # fallback if FPS cannot be read

frame_interval = max(1, int(round(src_fps / TARGET_FPS)))

frame_idx = 0
saved = 0

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
pbar = tqdm(total=total_frames, desc="Extracting frames")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        # Convert from BGR to RGB (models expect RGB)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        out_path = os.path.join(FRAME_DIR, f"frame_{saved:06d}.png")
        cv2.imwrite(out_path, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
        saved += 1
    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()
print(f"Extracted {saved} frames to '{FRAME_DIR}'.")


In [ ]:
import glob
import numpy as np
from PIL import Image
import torch
from depth_anything_3.api import DepthAnything3
from tqdm import tqdm

# Select device: use MPS on Apple Silicon if available, otherwise fallback to CPU.
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print("Using device:", device)

# Load the DA3METRIC‑LARGE model
MODEL_NAME = "depth-anything/DA3METRIC-LARGE"
model = DepthAnything3.from_pretrained(MODEL_NAME).to(device)
model.eval()

# Gather frame paths
image_paths = sorted(glob.glob(os.path.join(FRAME_DIR, "*.png")))
print("Number of frames to process:", len(image_paths))

# Output directories for raw depth arrays and visualization images
DEPTH_NPY_DIR = "depth_npy"
DEPTH_VIS_DIR = "depth_vis"
os.makedirs(DEPTH_NPY_DIR, exist_ok=True)
os.makedirs(DEPTH_VIS_DIR, exist_ok=True)

# Function to normalize depth for visualization
def normalize_depth(depth):
    depth = depth.astype(np.float32)
    valid = np.isfinite(depth)
    if valid.sum() == 0:
        return np.zeros(depth.shape, dtype=np.uint8)
    low = np.percentile(depth[valid], 2)
    high = np.percentile(depth[valid], 98)
    if high <= low:
        high = low + 1e-6
    vis = (depth - low) / (high - low)
    vis = np.clip(vis, 0, 1)
    return (vis * 255).astype(np.uint8)

# Process frames in small batches to manage memory on M2 MacBook
CHUNK_SIZE = 2  # reduce this (e.g., to 1) if you encounter memory errors

for start in tqdm(range(0, len(image_paths), CHUNK_SIZE), desc="Running depth inference"):
    batch_paths = image_paths[start:start + CHUNK_SIZE]
    with torch.no_grad():
        prediction = model.inference(batch_paths)
    depths = prediction.depth
    for i, depth in enumerate(depths):
        global_idx = start + i
        stem = f"frame_{global_idx:06d}"
        depth_array = np.asarray(depth, dtype=np.float32)
        # Save raw depth map
        np.save(os.path.join(DEPTH_NPY_DIR, f"{stem}_depth.npy"), depth_array)
        # Save visualization
        vis_img = normalize_depth(depth_array)
        Image.fromarray(vis_img).save(os.path.join(DEPTH_VIS_DIR, f"{stem}_depth_vis.png"))

print("Depth inference complete.")


In [ ]:
# Optional: Visualize a few depth map previews
import matplotlib.pyplot as plt

preview_paths = sorted(glob.glob(os.path.join(DEPTH_VIS_DIR, "*.png")))[:3]

for p in preview_paths:
    img = Image.open(p)
    plt.figure(figsize=(5, 4))
    plt.imshow(img, cmap='gray')
    plt.title(os.path.basename(p))
    plt.axis('off')
    plt.show()


In [ ]:
# Package results into ZIP archives (optional)
import shutil

zip_npy = shutil.make_archive("depth_npy_results", "zip", DEPTH_NPY_DIR)
zip_vis = shutil.make_archive("depth_vis_results", "zip", DEPTH_VIS_DIR)

print("Created archives:")
print("Raw depth maps:", zip_npy)
print("Visual depth maps:", zip_vis)
